<a href="https://colab.research.google.com/github/riyoprayogi/DeepLearn/blob/main/Chest_X_Ray.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Data:**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
save_dir = '/content/drive/MyDrive/Tubes_DL-Kel2'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
!pip install transformers torch torchvision opencv-python matplotlib pandas nltk -q

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
import xml.etree.ElementTree as ET

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset, random_split

import torchvision.models as models
from torchvision import transforms
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, GPT2LMHeadModel


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Library berhasil dimuat. Menggunakan device: {device}")

In [ ]:
# Setup Kaggle API
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
 files.upload()
 !mkdir -p ~/.kaggle
 !cp kaggle.json ~/.kaggle/
 !chmod 600 ~/.kaggle/kaggle.json
# Download & Extract
!kaggle datasets download -d reemaalbishri/indiana-university-chest-x-rays
!unzip -q indiana-university-chest-x-rays.zip -d /content/iu_data

### Ekstraksi File Data

Sel ini mengekstrak file gambar X-ray (format PNG) dan laporan radiologi (format XML) dari arsip `.tgz` yang telah diunduh. File-file ini penting untuk pelatihan model.

In [ ]:

# Ekstrak file gambar
!tar -xzf /content/iu_data/NLMCXR_png.tgz -C /content/iu_data/

# Ekstrak file teks/report
!tar -xzf /content/iu_data/NLMCXR_reports.tgz -C /content/iu_data/

print("Ekstraksi selesai")
print(os.listdir('/content/iu_data/'))

#  tahap Preprocecing

Sel ini memparsing ribuan file XML laporan radiologi untuk mengekstrak 'Findings' (temuan) dan 'Impression' (kesan). Data yang diekstrak kemudian digabungkan dengan nama gambar yang sesuai dan disimpan dalam format CSV untuk memudahkan pemrosesan lebih lanjut.

In [ ]:
xml_dir   = '/content/iu_data/ecgen-radiology'
data_list = []

print("Membaca file XML...")
for filename in os.listdir(xml_dir):
    if not filename.endswith('.xml'):
        continue
    try:
        tree = ET.parse(os.path.join(xml_dir, filename))
        root = tree.getroot()
        uid        = filename.replace('.xml', '')
        findings   = ""
        impression = ""

        for abstract in root.findall('.//AbstractText'):
            label = abstract.get('Label')
            text  = abstract.text if abstract.text else ""
            if label == 'FINDINGS':
                findings = text
            elif label == 'IMPRESSION':
                impression = text

        for parent_image in root.findall('.//parentImage'):
            image_id = parent_image.get('id')
            if image_id:
                data_list.append({
                    'uid':        uid,
                    'image_name': image_id + '.png',
                    'findings':   findings,
                    'impression': impression
                })
    except Exception:
        pass

df = pd.DataFrame(data_list)
df['report_check'] = df['findings'].fillna('') + df['impression'].fillna('')
df = df[df['report_check'].str.strip() != ''].drop(columns=['report_check'])

csv_path = '/content/iu_data/indiana_reports.csv'
df.to_csv(csv_path, index=False)
print(f"✅ CSV dibuat: {len(df)} baris → '{csv_path}'")

In [ ]:
IMAGE_DIR = None
for candidate in ['/content/iu_data/images',
                  '/content/iu_data/NLMCXR_png',
                  '/content/iu_data']:
    if os.path.isdir(candidate):
        pngs = [f for f in os.listdir(candidate) if f.endswith('.png')]
        if pngs:
            IMAGE_DIR = candidate
            print(f" Folder gambar: {IMAGE_DIR} ({len(pngs)} file PNG)")
            break

if IMAGE_DIR is None:
    raise RuntimeError(" Folder gambar tidak ditemukan. Periksa hasil ekstraksi.")

# Tampilan Data Mentah

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

IMAGE_DIR = None
for candidate in ['/content/iu_data/images',
                  '/content/iu_data/NLMCXR_png',
                  '/content/iu_data']:
    if os.path.isdir(candidate):
        pngs = [f for f in os.listdir(candidate) if f.endswith('.png')]
        if pngs:
            IMAGE_DIR = candidate
            print(f" Folder gambar: {IMAGE_DIR} ({len(pngs)} file PNG)")
            break

if IMAGE_DIR is None:
    raise RuntimeError(" Folder gambar tidak ditemukan. Periksa hasil ekstraksi.")

# Get 3 random image paths from the dataframe
random_rows = df.sample(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Raw X-Ray Images', fontsize=16)

for i, (index, row) in enumerate(random_rows.iterrows()):
    image_name = row['image_name']
    image_path = os.path.join(IMAGE_DIR, image_name)

    # Load and display the image
    print(f"Displaying image: {image_name}")
    image = Image.open(image_path)
    axes[i].imshow(image)
    axes[i].set_title(f'Image: {image_name}')
    axes[i].axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

### Definisi Kelas Dataset (IUXRayDataset)

Sel ini mendefinisikan kelas `IUXRayDataset` yang bertanggung jawab untuk memuat gambar X-ray dan teks laporan. Kelas ini juga menangani tokenisasi teks dan transformasi gambar, memastikan data siap untuk model pembelajaran mendalam.

In [ ]:
class IUXRayDataset(Dataset):
    """
    Dataset IU X-Ray.
    Menggabungkan findings + impression menjadi satu laporan,
    lalu memvalidasi bahwa file gambar benar-benar ada di disk.
    """
    def __init__(self, csv_path, img_dir, tokenizer, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.img_dir   = img_dir
        self.tokenizer = tokenizer
        self.transform = transform

        # Gabungkan findings + impression
        self.df['report'] = (
            self.df['findings'].fillna('') + " " +
            self.df['impression'].fillna('')
        ).str.strip()

        # Hapus baris tanpa teks laporan
        self.df = self.df[self.df['report'] != ""].reset_index(drop=True)

        # BUG FIX: Validasi file gambar ada di disk sebelum training
        valid_mask = self.df['image_name'].apply(
            lambda x: os.path.exists(os.path.join(self.img_dir, str(x)))
        )
        self.df = self.df[valid_mask].reset_index(drop=True)
        print(f"   Dataset: {len(self.df)} sampel valid (gambar ada di disk)")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, str(row['image_name']))

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        tokens = self.tokenizer(
            row['report'],
            padding='max_length',
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        return {
            'image':         image,
            'input_ids':     tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0)
        }

### Konfigurasi Tokenizer dan Transformasi Gambar

Sel ini mengonfigurasi `tokenizer` dari model DistilGPT2 untuk memproses teks laporan dan `transform` untuk mengubah ukuran dan menormalisasi gambar X-ray. Ini adalah langkah penting untuk memastikan konsistensi input ke model.

In [ ]:
# 1. Gunakan DistilGPT2 Tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

# 2. aturResolusi 224x224
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### Inisialisasi DataLoader

Sel ini membuat `DataLoader` yang akan memuat data dalam batch selama pelatihan. Dataset dipotong menjadi `dataset_demo`  untuk tujuan demonstrasi dan pengujian cepat.

In [ ]:
print("Memuat dataset...")
dataset_full = IUXRayDataset(csv_path, IMAGE_DIR, tokenizer, transform)

N_DEMO       = min(500, len(dataset_full))
dataset_demo = Subset(dataset_full, range(N_DEMO))

# Train / Val split  80:20
train_size                  = int(0.8 * len(dataset_demo))
val_size                    = len(dataset_demo) - train_size
train_dataset, val_dataset  = random_split(dataset_demo, [train_size, val_size])

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"\n DataLoader siap!")
print(f"   Train : {len(train_dataset)} sampel | {len(train_loader)} batch")
print(f"   Val   : {len(val_dataset)} sampel | {len(val_loader)} batch")

### Arsitektur Model MLRG_WeaklySupervised_Heatmap

Sel ini mendefinisikan arsitektur model `MLRG_WeaklySupervised_Heatmap`. Model ini terdiri dari:

1.  **Vision Encoder (resNet50):** Untuk mengekstrak fitur dari gambar X-ray.
2.  **Spatial Attention (Heatmap Generator):** Untuk menghasilkan heatmap yang menunjukkan area penting pada gambar.
3.  **Text Decoder (DistilGPT2):** Untuk menghasilkan teks laporan berdasarkan fitur gambar dan heatmap.
4.  **Proyeksi:** Untuk menyesuaikan dimensi fitur gambar agar kompatibel dengan model teks.

In [ ]:
class MLRG_WeaklySupervised_Heatmap(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # 1. Vision Encoder — ResNet50 tanpa avg-pool & FC terakhir
        resnet         = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.encoder   = nn.Sequential(*list(resnet.children())[:-2])
        # Output shape: (B, 2048, 7, 7)

        # 2. Spatial Attention — menghasilkan heatmap (B, 1, 7, 7)
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(512,    1, kernel_size=1),
            nn.Sigmoid()
        )

        # 3. Text Decoder — DistilGPT2 + cross-attention
        self.decoder = GPT2LMHeadModel.from_pretrained(
            'distilgpt2',
            add_cross_attention=True,
            ignore_mismatched_sizes=True
        )
        self.decoder.resize_token_embeddings(vocab_size)

        # 4. Proyeksi: 2048 → 768 (dimensi GPT-2)
        self.proj = nn.Linear(2048, 768)


    def forward(self, images, input_ids, attention_mask=None):
        # Vision encoding
        features = self.encoder(images)

        # Heatmap generation
        heatmap = self.spatial_attention(features)
        attended_features = features * heatmap

        # Reshape untuk GPT-2 cross-attention
        B, C, H, W  = attended_features.size()
        enc_hidden  = attended_features.view(B, C, H*W).permute(0, 2, 1)
        enc_hidden  = self.proj(enc_hidden)

        # Text generation + loss
        outputs = self.decoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids,
            encoder_hidden_states=enc_hidden
        )

        return outputs.loss, outputs.logits, heatmap

### Training loop Model

Sel ini memulai proses pelatihan model. Model dilatih selama 3 epoch, dengan menghitung loss dan memperbarui bobot model menggunakan optimizer AdamW. Setiap model yang terlatih per epoch disimpan.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = MLRG_WeaklySupervised_Heatmap(len(tokenizer)).to(device)

# Freeze encoder ResNet50 → hemat VRAM ~3x lebih cepat
for param in model.encoder.parameters():
    param.requires_grad = False

trainable_params = (
    list(model.spatial_attention.parameters()) +
    list(model.proj.parameters()) +
    list(model.decoder.parameters())
)
optimizer = torch.optim.AdamW(trainable_params, lr=5e-5, weight_decay=1e-4)
scaler    = torch.cuda.amp.GradScaler()

epochs   = 3
history  = {'train_loss': [], 'val_loss': []}

print(f"\n Mulai training ({epochs} epoch)...\n")

for epoch in range(epochs):
    # TRAIN
    model.train()
    total_train_loss = 0.0


    for i, batch in enumerate(train_loader):
        images         = batch['image'].to(device)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)  # ambil dari batch

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            loss, logits, heatmap = model(images, input_ids, attention_mask)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_train_loss += loss.item()

        if i % 10 == 0:
            print(f"  Epoch [{epoch+1}/{epochs}] Step [{i}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}")

    avg_train = total_train_loss / len(train_loader)
    history['train_loss'].append(avg_train)

    # VALIDASI
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images         = batch['image'].to(device)
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            loss, _, _     = model(images, input_ids, attention_mask)
            total_val_loss += loss.item()

    avg_val = total_val_loss / len(val_loader)
    history['val_loss'].append(avg_val)

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
    print(f"{'='*60}\n")

    # Simpan checkpoint ke Drive
    ckpt_path = os.path.join(save_dir, f'mlrg_epoch_{epoch+1}.pth')
    torch.save({
        'epoch':                epoch + 1,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss':           avg_train,
        'val_loss':             avg_val,
    }, ckpt_path)
    print(f"✅ Checkpoint disimpan: {ckpt_path}\n")

# Kurva Lose

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, epochs+1), history['train_loss'],
         marker='o', label='Train Loss', color='steelblue')
plt.plot(range(1, epochs+1), history['val_loss'],
         marker='s', label='Val Loss',   color='coral')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'loss_curve.png'), dpi=150)
plt.show()

### Visualisasi Hasil Inferensi dan Heatmap

Sel ini mendefinisikan dan menjalankan fungsi `visualize_inference`. Fungsi ini mengambil satu batch data dari DataLoader, melakukan forward pass melalui model untuk mendapatkan prediksi teks dan heatmap. Kemudian, ia menampilkan:

1.  **Gambar X-ray asli**.
2.  **Gambar X-ray dengan overlay heatmap**, menunjukkan area yang menjadi fokus perhatian model saat menghasilkan laporan.
3.  **Perbandingan antara laporan radiologi ground truth (asli) dan laporan yang diprediksi oleh model AI**.

In [ ]:
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)


def visualize_with_colorbar(model, val_loader, tokenizer, device, save_path=None):
    model.eval()

    # ── FIX: Kumpulkan semua data, lalu pilih random ──────────────
    all_images, all_input_ids = [], []
    for batch in val_loader:
        all_images.append(batch['image'])
        all_input_ids.append(batch['input_ids'])

    all_images    = torch.cat(all_images,    dim=0)
    all_input_ids = torch.cat(all_input_ids, dim=0)

    # Pilih 1 sampel secara random setiap kali dijalankan
    idx = torch.randint(0, len(all_images), (1,)).item()
    print(f"Sampel index: {idx}")

    images    = all_images[idx:idx+1].to(device)
    input_ids = all_input_ids[idx:idx+1].to(device)
    # ──────────────────────────────────────────────────────────────

    with torch.no_grad():
        _, logits, heatmaps = model(images, input_ids=input_ids)

    img   = denormalize(images[0].cpu()).permute(1, 2, 0).numpy()
    hm    = heatmaps[0, 0].cpu().numpy()
    hm    = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)
    hm_up = cv2.resize(hm, (224, 224), interpolation=cv2.INTER_CUBIC)

    pred_text = tokenizer.decode(
        torch.argmax(logits[0], dim=-1), skip_special_tokens=True
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle("MLRG+ Chest X-Ray Analysis", fontsize=14, fontweight='bold')

    axes[0].imshow(img)
    axes[0].set_title("Input X-Ray", fontsize=12)
    axes[0].axis('off')

    im = axes[1].imshow(hm_up, cmap='jet')
    axes[1].set_title("Attention Heatmap (Raw)", fontsize=12)
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046)

    axes[2].imshow(img)
    axes[2].imshow(hm_up, cmap='jet', alpha=0.45)
    axes[2].set_title("Overlay: Region of Interest", fontsize=12)
    axes[2].axis('off')

    # Generated report tetap muncul di bawah
    fig.text(0.5, -0.02,
             f"Generated Report: {pred_text[:180]}",
             ha='center', fontsize=9, style='italic',
             bbox=dict(boxstyle='round', facecolor='lightyellow'))

    plt.tight_layout()
    out_path = save_path or os.path.join(save_dir, f'mlrg_result_idx{idx}.png')
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✅ Gambar disimpan: {out_path}")


# Jalankan — setiap run akan tampil gambar berbeda
visualize_with_colorbar(model, val_loader, tokenizer, device)

# Visualisasi lengkap dengan report

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import textwrap
import cv2
import torch
import numpy as np

def visualisasi_final_report(model, val_loader, tokenizer, device, save_path=None):
    model.eval()


    batch = next(iter(val_loader))
    images = batch['image'].to(device)
    input_ids = batch['input_ids'].to(device)

    img_tensor = images[0:1]
    id_tensor  = input_ids[0:1]

    with torch.no_grad():
        _, logits, heatmaps = model(img_tensor, input_ids=id_tensor)


    img = denormalize(img_tensor[0].cpu()).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)

    hm = heatmaps[0, 0].cpu().numpy()
    hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)
    hm_up = cv2.resize(hm, (224, 224), interpolation=cv2.INTER_CUBIC)

    ground_truth = tokenizer.decode(id_tensor[0], skip_special_tokens=True)
    pred_ids     = torch.argmax(logits[0], dim=-1)
    prediction   = tokenizer.decode(pred_ids, skip_special_tokens=True)

    wrapper = textwrap.TextWrapper(width=60)
    gt_wrapped = wrapper.fill(ground_truth)
    pred_wrapped = wrapper.fill(prediction)


    fig = plt.figure(figsize=(18, 11), facecolor='white')
    # Membuat grid: 2 baris utama (Gambar & Teks)
    gs = gridspec.GridSpec(2, 3, height_ratios=[1.2, 1], hspace=0.3)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(img)
    ax1.set_title("1. Input X-Ray", fontsize=13, fontweight='bold', pad=10)
    ax1.axis('off')

    ax2 = fig.add_subplot(gs[0, 1])
    im = ax2.imshow(hm_up, cmap='jet')
    ax2.set_title("2. Attention Heatmap", fontsize=13, fontweight='bold', pad=10)
    ax2.axis('off')
    plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(img)
    ax3.imshow(hm_up, cmap='jet', alpha=0.45)
    ax3.set_title("3. Overlay: Region of Interest", fontsize=13, fontweight='bold', pad=10)
    ax3.axis('off')


    # Reference Report
    ax4 = fig.add_subplot(gs[1, 0:1])
    ax4.axis('off')
    ax4.text(0, 1.1, "📋 REFERENCE REPORT", fontsize=14, fontweight='bold', color='#2c3e50')
    ax4.text(0, 0.95, gt_wrapped, fontsize=11, va='top', ha='left', wrap=True,
             bbox=dict(boxstyle='round,pad=1.2', facecolor='#f8f9fa', edgecolor='#dee2e6', alpha=1))

    # Generated Report
    ax5 = fig.add_subplot(gs[1, 1:3])
    ax5.axis('off')
    ax5.text(0, 1.1, "🤖 GENERATED REPORT", fontsize=14, fontweight='bold', color='#d35400')
    ax5.text(0, 0.95, pred_wrapped, fontsize=11, va='top', ha='left', wrap=True,
             bbox=dict(boxstyle='round,pad=1.2', facecolor='#fff9f0', edgecolor='#ffe8cc', alpha=1))


    plt.savefig(save_path or 'final_medical_report.png', dpi=300, bbox_inches='tight')
    plt.show()

# Panggil fungsi
visualisasi_final_report(model, val_loader, tokenizer, device)

# Skor BLEU (Bilingual Evaluation Understudy)
Karena model menghasilkan teks (Generated Report), kita tidak bisa sekadar membandingkan "Benar" atau "Salah" seperti pilihan ganda. Kita menggunakan metrik BLEU Score

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def evaluate_bleu(model, val_loader, tokenizer, device):
    model.eval()
    references, hypotheses = [], []

    with torch.no_grad():
        for batch in val_loader:
            images    = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)

            _, logits, _ = model(images, input_ids=input_ids)

            for i in range(images.size(0)):
                ref_text    = tokenizer.decode(input_ids[i], skip_special_tokens=True)
                ref_tokens  = ref_text.lower().split()

                pred_ids    = torch.argmax(logits[i], dim=-1)
                pred_text   = tokenizer.decode(pred_ids, skip_special_tokens=True)
                pred_tokens = pred_text.lower().split()

                if ref_tokens and pred_tokens:
                    references.append([ref_tokens])
                    hypotheses.append(pred_tokens)

    smoother = SmoothingFunction().method1
    bleu1 = corpus_bleu(references, hypotheses,
                        weights=(1, 0, 0, 0), smoothing_function=smoother)
    bleu4 = corpus_bleu(references, hypotheses,
                        weights=(.25, .25, .25, .25), smoothing_function=smoother)
    return bleu1, bleu4


bleu1, bleu4 = evaluate_bleu(model, val_loader, tokenizer, device)
print(f"\n Evaluation Results:")
print(f"   BLEU-1 : {bleu1:.4f}")
print(f"   BLEU-4 : {bleu4:.4f}")